# 03 - Embedding Models (Hue Foods RAG MVP)

Notebook này trình bày Phase 3: dense embedding và sparse representation cho 572 canonical food chunks. Dense vectors biểu diễn ngữ nghĩa (tương đồng khái niệm), sparse vectors giữ tín hiệu từ khóa qua TF-IDF. Cả hai là input cho Phase 4 Qdrant ingestion.

Không có live model/API/web call nào trong default mode. Real-mode local E5 chỉ chạy khi opt-in bằng `HUE_RAG_LOCAL_E5=1` và model đã có trong local Hugging Face cache.


In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")


## Cấu hình embedding (`settings.yaml`)

`core/settings_loader.py` đọc `backend/config/settings.yaml`. Nhóm `embedding` khai báo local baseline: provider, model ID, dimension, device, batch_size và E5 instruction prefixes (`passage:` cho documents, `query:` cho queries). Nhóm `remote` khai báo OpenRouter adapter live-ready nhưng không bao giờ kích hoạt mặc định.

Expected output: `provider: sentence_transformer`, `model: intfloat/multilingual-e5-small`, `vector_size: 384`, `batch_size: 64` và đúng hai prefixes.


In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
embedding = settings["embedding"]
print("provider:", embedding["provider"])
print("model:", embedding["model"])
print("vector_size:", embedding["vector_size"])
print("device:", embedding["device"])
print("batch_size:", embedding["batch_size"])
print("document_prefix:", repr(embedding["document_prefix"]))
print("query_prefix:", repr(embedding["query_prefix"]))
print("remote model (inactive):", embedding["remote"]["model"])


## Dense embedding - interface và safe default mode

`BaseEmbedder` là interface tối thiểu cho consumers: `model_id`, `dimension`, `embed_documents(texts)` và `embed_query(query)`. Output luôn là L2-normalized vectors, finite và đúng dimension; dimension mismatch fail-fast, không pad/truncate và không fallback sang model khác.

Default mode dùng `FakeEmbedder` (subclass định nghĩa trong notebook) trả deterministic vectors - minh họa contract mà không load model. `embed_in_batches` giới hạn kích thước batch và giữ nguyên thứ tự input.


In [ ]:
import zlib

import numpy as np

from embedding.base import BaseEmbedder
from embedding.batch_embed import embed_in_batches


class FakeEmbedder(BaseEmbedder):
    """Deterministic fake embedder used as the safe default mode."""

    def __init__(self, model_id="fake/demo-embedder", dimension=384):
        self._model_id = model_id
        self._dimension = dimension

    def _raw(self, texts):
        # Deterministic per text so batched and direct embedding agree.
        return np.asarray(
            [
                np.random.default_rng(zlib.crc32(t.encode("utf-8"))).standard_normal(
                    self._dimension
                )
                for t in texts
            ]
        )

    @property
    def model_id(self):
        return self._model_id

    @property
    def dimension(self):
        return self._dimension

    def embed_documents(self, texts):
        return self._process_vectors(self._raw(texts))

    def embed_query(self, query):
        self._validate_query(query)
        return self._process_vectors(self._raw([query]))[0]


fake = FakeEmbedder()
texts = [
    "Bún bò Huế với nước dùng cay nồng.",
    "Cơm hến trộn rau sống và bánh tráng.",
    "Chè heo quay ngọt thanh mát.",
]
vectors = fake.embed_documents(texts)
query_vector = fake.embed_query("Quán bún bò ngon nhất ở Huế?")

print("model_id:", fake.model_id)
print("dimension:", fake.dimension)
print("document vectors:", len(vectors), "| query vector:", len(query_vector))
print("norms:", np.linalg.norm(vectors, axis=1).round(6))
print("query norm:", round(float(np.linalg.norm(query_vector)), 6))

batched = embed_in_batches(fake, texts, batch_size=2)
print("batched count:", len(batched), "| order preserved:", batched == vectors)


## Sparse representation - TF-IDF

`SparseEmbedder` token hóa (lowercase, bỏ ký tự không phải word, giữ Unicode tiếng Việt), fit vocabulary và document frequency từ corpus theo thứ tự deterministic (mỗi token tăng DF đúng một lần mỗi document), rồi encode theo công thức:

```text
idf(term) = log((num_documents + 1) / (document_frequency + 1)) + 1
value = term_frequency * idf(term)
```

Indices là vị trí token trong vocabulary; unknown tokens bị bỏ qua; empty text trả hai lists rỗng; encode trước fit bị reject.


In [ ]:
import math

from embedding.sparse_embedder import SparseEmbedder, tokenize

corpus = ["Bún bò Huế", "Cơm hến", "Bánh ép mè xửng", "Bún bò chay"]
sparse = SparseEmbedder().fit(corpus)

print("vocabulary size:", sparse.vocabulary_size)
print("num_documents:", sparse.num_documents)
print("tokens:", tokenize("Bún bò Huế, Cơm hến!"))
result = sparse.encode("Bún bò bò")
print("encode('Bún bò bò'):", result)
# df("bún") = df("bò") = 2 over 4 docs -> idf = log((4+1)/(2+1)) + 1
idf_bun = math.log((4 + 1) / (2 + 1)) + 1
expected_bun = 1 * idf_bun   # tf("bún") = 1
expected_bo = 2 * idf_bun    # tf("bò") = 2
print("expected values:", [round(expected_bun, 6), round(expected_bo, 6)])
match = (
    abs(result["values"][0] - expected_bun) < 1e-9
    and abs(result["values"][1] - expected_bo) < 1e-9
)
print("match:", match)


## Sparse trên 572 canonical chunks

Fit lại toàn bộ canonical corpus từ `chunk_foods_markdown()` (Phase 2). Vocabulary phải non-empty, và cùng corpus phải tái tạo cùng kết quả encode (deterministic).


In [ ]:
from ingestion.chunking.markdown_chunker import chunk_foods_markdown

chunks = chunk_foods_markdown()
texts = [chunk["text"] for chunk in chunks]
print("canonical chunks:", len(texts))

sparse_corpus = SparseEmbedder().fit(texts)
print("vocabulary size:", sparse_corpus.vocabulary_size)
print("num_documents:", sparse_corpus.num_documents)

first = SparseEmbedder().fit(texts).encode("bún bò huế")
second = SparseEmbedder().fit(texts).encode("bún bò huế")
print("deterministic across refits:", first == second)

sample = sparse_corpus.encode("Bún bò Huế ở quán nào ngon?")
print("sample indices:", sample["indices"][:8])
print("sample values:", [round(v, 3) for v in sample["values"][:8]])


## Real-mode: local E5 từ cache (opt-in)

Đặt `HUE_RAG_LOCAL_E5=1` khi chạy cell dưới để load `intfloat/multilingual-e5-small` từ local Hugging Face cache (offline, không download). Cell này đo latency và peak RSS của sample batch. OpenRouter adapter (`OpenRouterEmbedder`) không bao giờ chạy trong notebook này; mọi live embedding run cần user approval riêng.


In [ ]:
import os
import time

USE_LOCAL_E5 = os.environ.get("HUE_RAG_LOCAL_E5") == "1"
if USE_LOCAL_E5:
    os.environ["HF_HUB_OFFLINE"] = "1"  # force cached model, no download
    from embedding.embedder import SentenceTransformerEmbedder

    emb = SentenceTransformerEmbedder(
        settings["embedding"]["model"],
        settings["embedding"]["vector_size"],
        device=settings["embedding"]["device"],
        batch_size=settings["embedding"]["batch_size"],
        document_prefix=settings["embedding"]["document_prefix"],
        query_prefix=settings["embedding"]["query_prefix"],
    )
    sample = [
        "Bún bò Huế với nước dùng cay nồng.",
        "Cơm hến trộn rau sống và bánh tráng.",
    ]
    started = time.perf_counter()
    vectors = emb.embed_documents(sample)
    elapsed = time.perf_counter() - started
    query_vector = emb.embed_query("Quán bún bò ngon nhất ở Huế?")

    import resource

    import numpy as np

    print("model_id:", emb.model_id)
    print("dimension:", emb.dimension)
    print("document norms:", np.linalg.norm(vectors, axis=1).round(6))
    print("query dimension:", len(query_vector))
    print(f"latency for {len(sample)} docs: {elapsed:.3f}s")
    print(
        "peak RSS (MiB):",
        resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
    )
else:
    print(
        "Real mode disabled. Run this notebook with HUE_RAG_LOCAL_E5=1 to load "
        "the cached local E5 model from disk (offline, no network)."
    )


## Checklist xác nhận Phase 3

1. `backend on path` trỏ đúng thư mục `backend/`.
2. Embedding config hiển thị `intfloat/multilingual-e5-small`, 384 dimensions, `batch_size: 64` và đúng hai prefixes.
3. Fake embedder trả đúng 1 vector cho mỗi text cùng thứ tự, dimension 384, norm ≈ 1.0; `embed_in_batches` giữ thứ tự và số lượng.
4. Sparse sample: vocabulary size hợp lý, indices/values cùng length, values dương; giá trị TF-IDF khớp tính tay.
5. Corpus 572 chunks: vocabulary non-empty, deterministic giữa hai lần fit; sample indices/values hợp lệ.
6. Real-mode (tùy chọn): với `HUE_RAG_LOCAL_E5=1`, model từ cache trả 384-d normalized vectors, ghi latency và peak RSS; không có network.
7. Không có live API/model call nào trong default mode; không có key nào được sử dụng.
